# Capítulo 3: Plotagem e visualização de seus dados com `matplotlib`

**Chris Holden (ceholden@gmail.com) - [https://github.com/ceholden](https://github.com/ceholden)**

-----

## Introdução

[`matplotlib`](https://www.google.com/search?q=%5Bhttp://matplotlib.org/%5D\(http://matplotlib.org/\)) é uma biblioteca de plotagem muito poderosa para criar visualizações incríveis para publicações, uso pessoal ou até mesmo para aplicações *web* e *desktop*. O `matplotlib` pode gerar praticamente qualquer visualização bidimensional que você possa imaginar, incluindo histogramas, gráficos de dispersão, gráficos bivariados e exibição de imagens. Para se inspirar, confira a [galeria de exemplos](http://matplotlib.org/gallery.html) do `matplotlib`, que inclui o código-fonte necessário para gerar cada visualização.

Um ótimo recurso para aprender `matplotlib` é o material de [J.R. Johansson](https://github.com/jrjohansson/scientific-python-lectures).

## API do Matplotlib: Máquina de Estados *versus* Orientada a Objetos

Um aspecto do `matplotlib` que pode ser inicialmente confuso é que ele oferece duas abordagens principais para a plotagem: o método orientado a objetos e o método de máquina de estados.

Embora a biblioteca possa ser usada de maneira orientada a objetos (onde você cria um objeto que representa a figura, e a figura, por sua vez, cria objetos que representam os eixos, etc.), o uso mais familiar para usuários de MATLAB é o ambiente de máquina de estados [`pyplot`](https://www.google.com/search?q=%5Bhttp://matplotlib.org/api/pyplot_api.html%5D\(http://matplotlib.org/api/pyplot_api.html\)):

> "O ambiente de máquina de estados do Pyplot se comporta de forma semelhante ao MATLAB e deve ser mais familiar para usuários com experiência em MATLAB."

Em geral, você deve usar o método `Pyplot` (máquina de estados) ao plotar dados interativamente ou ao desenvolver visualizações rápidas. A API orientada a objetos, embora mais complexa, é uma maneira muito mais poderosa de criar gráficos e deve ser usada ao desenvolver visualizações mais complexas.

Como esta é uma breve introdução ao `matplotlib`, usaremos o método de máquina de estados `Pyplot` para criar visualizações.

## Configuração do Colab e Preparação dos Dados

Primeiro, vamos garantir que as bibliotecas estejam instaladas e carregar os dados, repetindo o processo dos capítulos anteriores:

In [ ]:
# Instala as bibliotecas (repetido para garantir o ambiente no Colab)
!pip install gdal matplotlib --quiet

# Importa os submódulos necessários
from osgeo import gdal
from osgeo import gdal_array
import numpy as np

# --- ATENÇÃO (Substitua o caminho abaixo pelo caminho do seu arquivo raster no Colab) ---
# Se o arquivo não estiver carregado, a execução desta célula pode falhar ou 'dataset' será None.
dataset = None
try:
    # Use um caminho de arquivo de exemplo se estiver disponível
    # dataset = gdal.Open('/content/LE70220491999322EDC01_stack.gtif', gdal.GA_ReadOnly)

    # Se você está apenas executando o notebook, pode ser necessário simular os arrays:
    if dataset is None:
        print("Dataset não carregado. Criando dados de exemplo para visualização...")
        # Criando dados fictícios para a demonstração
        rows, cols, num_bands = 250, 250, 7
        image = np.random.randint(0, 10000, size=(rows, cols, num_bands), dtype=np.uint16)

        # Simula as bandas Vermelha (índice 2) e NIR (índice 3)
        red_data = image[:, :, 2]
        nir_data = image[:, :, 3]

        # Garante que a divisão seja feita com ponto flutuante para o NDVI
        ndvi = (nir_data.astype(np.float64) - red_data) / (nir_data + red_data)

    else:
        # Carrega os dados reais do dataset
        image_datatype = dataset.GetRasterBand(1).DataType
        image = np.zeros((dataset.RasterYSize, dataset.RasterXSize, dataset.RasterCount),
                         dtype=gdal_array.GDALTypeCodeToNumericTypeCode(image_datatype))

        for b in range(dataset.RasterCount):
            band = dataset.GetRasterBand(b + 1)
            image[:, :, b] = band.ReadAsArray()

        # Calcula o NDVI (assumindo NIR no índice 3 e Red no índice 2)
        ndvi = (image[:, :, 3].astype(np.float64) - image[:, :, 2]) / \
                (image[:, :, 3] + image[:, :, 2])

        dataset = None # Fecha o dataset
        print("Dados carregados e NDVI calculado.")

except Exception as e:
    print(f"Erro ao carregar ou criar dados: {e}")
    image = None
    ndvi = None

Com os dados lidos e o NDVI calculado (ou simulado), vamos fazer algumas plotagens.

## Plotagem Básica

Primeiro, importamos o `matplotlib` para o nosso *namespace*. Usaremos um recurso especial do ambiente Jupyter/Colab que permite incorporar as figuras do `matplotlib` diretamente no *notebook* (inline), usando o comando mágico `%matplotlib inline`.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

Com o `matplotlib` importado, podemos criar uma figura e fazer nosso primeiro [gráfico](http://matplotlib.org/api/pyplot_api.html#matplotlib.pyplot.plot):

In [ ]:
# Array de 0 a 9
x = np.arange(10)
# 10 números aleatórios, entre 0 e 10
y = np.random.randint(0, 10, size=10)

# plota-os como linhas
plt.plot(x, y)

In [ ]:
# Plota-os apenas como pontos -- especifique "ls" ("linestyle") como uma string nula
plt.plot(x, y, 'ro', ls='')

## Plotagem de *Arrays* 2D: Gráficos de Dispersão (Scatterplots)

Uma coisa típica que podemos querer fazer é plotar uma banda contra a outra. Para isso, precisamos transformar, ou **achatar** (*flatten*), nossos *arrays* 2D de cada banda em *arrays* 1D:

In [ ]:
if image is not None:
    # Acessa as bandas Vermelha (índice 2) e NIR (índice 3)
    red_band = image[:, :, 2]
    nir_band = image[:, :, 3]

    print('Formato do Array antes: {shp} (tamanho é {sz})'.format(shp=red_band.shape, sz=red_band.size))

    # Achata os arrays para 1D
    red = np.ndarray.flatten(red_band)
    nir = np.ndarray.flatten(nir_band)

    print('Formato do Array depois: {shp} (tamanho é {sz})'.format(shp=nir.shape, sz=nir.size))

Achamos o número de entradas em cada banda raster, mas as achatamos de 2 dimensões para 1.

Agora podemos plotá-las. Como queremos apenas pontos, podemos usar `scatter` para um [gráfico de dispersão](http://matplotlib.org/api/pyplot_api.html#matplotlib.pyplot.scatter) (*scatterplot*). Como não há linhas em um gráfico de dispersão, ele tem uma sintaxe ligeiramente diferente.

In [ ]:
if image is not None:
    # Cria o gráfico
    plt.scatter(red, nir, color='r', marker='o')

    # Adiciona rótulos aos eixos
    plt.xlabel('Reflectância do Vermelho (Red Reflectance)')
    plt.ylabel('Reflectância do NIR (NIR Reflectance)')

    # Adiciona um título
    plt.title('Gráfico de Dispersão Red vs NIR')

Se quisermos que os dois eixos tenham os mesmos limites (útil para analisar a "nuvem" de dados), podemos calcular os limites e aplicá-los:

In [ ]:
if image is not None:
    # Cria o gráfico
    plt.scatter(red, nir, color='r', marker='o', s=1) # s=1 para pontos menores

    # Calcula o mínimo e o máximo para ambos os eixos
    plot_min = min(red.min(), nir.min())
    plot_max = max(red.max(), nir.max())

    plt.xlim((plot_min, plot_max))
    plt.ylim((plot_min, plot_max))

    # Adiciona rótulos aos eixos
    plt.xlabel('Reflectância do Vermelho (Red Reflectance)')
    plt.ylabel('Reflectância do NIR (NIR Reflectance)')

    # Adiciona um título
    plt.title('Gráfico de Dispersão Red vs NIR (Limites Iguais)')

## Plotagem de *Arrays* 2D: Imagens

Com tantos dados disponíveis para visualização, pode ser difícil entender o que está acontecendo com a "nuvem" de pontos mostrada acima. Felizmente, nossos *datasets* possuem uma estrutura espacial.

Para mostrar a estrutura espacial de nossas imagens, podemos fazer uma plotagem de imagem de uma de nossas bandas usando `imshow` para [exibir uma imagem nos eixos](http://matplotlib.org/api/pyplot_api.html#matplotlib.pyplot.imshow):

In [ ]:
if image is not None:
    # NIR band (índice 3 na nossa simulação)
    plt.imshow(image[:, :, 3], cmap=plt.cm.Greys_r) # Greys_r inverte o mapa de cores (tons de cinza)
    plt.title('Banda Infravermelho Próximo (NIR)')

Bem, parece que algo está acontecendo, talvez um rio no centro e alguma vegetação brilhante no canto inferior esquerdo da imagem. O que falta é a identificação do significado das cores.

Felizmente, o `matplotlib` pode nos fornecer uma [barra de cores](http://matplotlib.org/api/pyplot_api.html#matplotlib.pyplot.colorbar) (*colorbar*).

In [ ]:
if image is not None:
    # NIR band (índice 3 na nossa simulação)
    plt.imshow(image[:, :, 3], cmap=plt.cm.Greys_r)
    plt.colorbar()
    plt.title('Banda NIR com Barra de Cores')

Se quisermos uma imagem em tons de cinza mais intuitiva, podemos especificar manualmente um [mapa de cores](http://matplotlib.org/api/colors_api.html#matplotlib.colors.Colormap) (*colormap*):

In [ ]:
if image is not None and ndvi is not None:
    # Plota o NIR na primeira subtela (tons de cinza)
    plt.subplot(121)
    plt.imshow(image[:, :, 3], cmap=plt.cm.Greys_r)
    plt.title('Banda NIR')
    plt.colorbar(orientation='horizontal')

    # Plota o NDVI na segunda subtela (tons de cinza invertido)
    plt.subplot(122)
    # NDVI geralmente é plotado em um esquema de cores que destaca vegetação (como 'viridis' ou 'Greens')
    # O cmap=plt.cm.Greys_r no original inverte os tons de cinza.
    plt.imshow(ndvi, cmap=plt.cm.Greens)
    plt.title('NDVI')
    plt.colorbar(orientation='horizontal')

    plt.tight_layout()

## Plotagem de *Arrays* 3D: Imagens Multiespectrais (Cor Verdadeira/Falsa)

Imagens em tons de cinza são agradáveis, mas a maior parte da informação que podemos receber vem da visualização da interação entre diferentes bandas. Para conseguir isso, podemos mapear diferentes bandas espectrais para os canais **Vermelho (Red), Verde (Green) e Azul (Blue)** (RGB) dos nossos monitores.

Antes de fazermos isso, o *help* do `matplotlib` `imshow` nos diz que precisamos [normalizar](http://en.wikipedia.org/wiki/Normalization_%28image_processing%29) nossas bandas para um intervalo de 0 a 1. Para isso, realizaremos um simples escalonamento linear, ajustando 0 de reflectância para 0 e 80% de reflectância (valor 8000 no dado original int16) para 1, cortando qualquer valor maior ou menor.

> **Lembre-se:** Se estivermos convertendo de um tipo de dado `Int16` (por exemplo, reflectância escalonada por 10.000x) para um decimal entre 0 e 1, precisaremos usar um tipo de dado **Float**\!

In [ ]:
if image is not None and ndvi is not None:
    # Extrai referências para as bandas SWIR1, NIR e Red.
    # Assumindo a ordem das bandas Landsat 7 no stack (típico: B1, B2, B3 (Red), B4 (NIR), B5 (SWIR1), B7 (SWIR2))
    # Para COR FALSA (NIR-SWIR1-RED):
    index_rgb = np.array([3, 4, 2])  # NIR (índice 3) -> R, SWIR1 (índice 4) -> G, Red (índice 2) -> B

    # Cria uma cópia e converte para float para o cálculo de normalização
    colors = image[:, :, index_rgb].astype(np.float64)

    # Valores de corte (equivalente a 0% a 80% de reflectância para dados 16-bit)
    max_val = 8000
    min_val = 0

    # Aplica o corte (clipping)
    colors[colors[:, :, :] > max_val] = max_val
    colors[colors[:, :, :] < min_val] = min_val

    # Normalização para o intervalo 0-1
    for b in range(colors.shape[2]):
        colors[:, :, b] = colors[:, :, b] * 1 / (max_val - min_val)

    # Plota a imagem em Cor Falsa (NIR-SWIR1-RED)
    plt.figure(figsize=(12, 6))
    plt.subplot(121)
    plt.imshow(colors)
    plt.title('Composição Colorida Falsa (NIR-SWIR1-Red)')

    # Plota o NDVI (repetido para referência)
    plt.subplot(122)
    plt.imshow(ndvi, cmap=plt.cm.Greens)
    plt.title('NDVI')
    plt.colorbar(orientation='horizontal', fraction=0.046, pad=0.04)

    plt.tight_layout()

## Conclusão

Vimos como o `matplotlib` pode ser combinado com o NumPy e o GDAL para visualizar e explorar facilmente nossos dados de sensoriamento remoto. No próximo capítulo, abordaremos como usar a biblioteca companheira do GDAL - **OGR** - para abrir e ler dados vetoriais.